# HashiCorp Vault -- exercices

## Set Up

### 1. Ajout de la clé GPG d'HashiCorp

Cette étape garantit que le logiciel que tu vas télécharger provient bien de l'éditeur et n'a pas été altéré.

```Bash
wget -O- https://apt.releases.hashicorp.com/gpg | sudo gpg --dearmor -o /usr/share/keyrings/hashicorp-archive-keyring.gpg
```

### 2. Ajout du dépôt officiel

Nous déclarons le dépôt d'HashiCorp dans les sources de ton gestionnaire de paquets.

```Bash
echo "deb [signed-by=/usr/share/keyrings/hashicorp-archive-keyring.gpg] https://apt.releases.hashicorp.com $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/hashicorp.list
```

### 3. Installation de Vault

Mise à jour de l'index et installation du binaire natif.

```Bash
sudo apt update && sudo apt install vault
```

### 4. Démarrage de l'infrastructure

Dans ton terminal système, lance la commande suivante et garde le Root Token affiché sous la main.

```Bash
vault server -dev
```

Ne ferme pas ce Terminal.
Installe les dépendances dans un autre terminal.

```Bash
uv add hvac requests
```

### 5. Variable d’environnement

L'architecture de configuration de Vault Agent accepte l'injection de paramètres via des fichiers, des drapeaux CLI, et des variables d'environnement par ordre de priorité croissante. Les scripts Python liront ces variables d'environnement (`VAULT_ADDR`, `VAULT_TOKEN`). Le client HVAC peut par exemple interroger le backend système pour lire le statut de leader de l'infrastructure afin de valider la connectivité :


In [ ]:
import os
import hvac

# 1. Définition éphémère pour le serveur de développement local
# Remplacer la valeur par le jeton "Root Token" fourni par le terminal Vault
os.environ["VAULT_ADDR"] = "http://127.0.0.1:8200"
os.environ["VAULT_TOKEN"] = (
    "hvs.MON_TOKEN_EPHEMERE_LOCAL"  # À remplacer avec le vrai token éphémére
)

# 2. Initialisation du client administratif HVAC
# Le client lit de manière standard les variables d'environnement
client = hvac.Client(url=os.getenv("VAULT_ADDR"), token=os.getenv("VAULT_TOKEN"))

# 3. Validation du statut du serveur
leader_status = client.sys.read_leader_status()
print(
    f"Connexion réussie. Le serveur est prêt (Leader HA: {leader_status['ha_enabled']})"
)

Connexion réussie. Le serveur est prêt (Leader HA: False)


### ⚠️ Note de Sécurité et d'Architecture : Gestion des Variables

Le document de référence précise explicitement que déployer des jetons Vault en dur dans le code d'une application est une faille de sécurité majeure. Dans une architecture de production réelle, le code Python doit être totalement agnostique : il se contente de lire les variables d'environnement (`VAULT_ADDR`, `VAULT_TOKEN`) préalablement injectées par l'infrastructure (via un fichier `.env`, l'orchestrateur de conteneurs, ou le démon Vault Agent).

**Choix pédagogique :** Puisque ce tutoriel s'appuie sur un serveur Vault local en mode développement (`-dev`), le jeton racine est volatile et change à chaque redémarrage. Pour fluidifier l'exécution de ces exercices, nous allons artificiellement injecter ce jeton dans l'environnement de la session Jupyter courante via le module `os` de Python. Ne reproduisez jamais ce schéma d'injection directe dans vos dépôts d'entreprise.


## Niveau 1 : Fondamentaux du Moteur Key/Value (KV-V2)

### 1. Activation, Configuration et Mécanisme CAS (Check-And-Set)

La première étape consiste à configurer le moteur de secrets. L'API sys/mounts/ permet de créer un point de montage et d'y attacher le plugin kv en spécifiant la version 2. Via HVAC, il est possible d'activer le moteur et d'appliquer immédiatement des règles de configuration strictes pour l'ensemble du chemin.


In [ ]:
import hvac

# 1. Activation sécurisée (Idempotente)
try:
    client.sys.enable_secrets_engine(
        backend_type="kv", path="shared", options={"version": "2"}
    )
    print("Moteur KV-v2 activé sur le chemin 'shared/'.")
except hvac.exceptions.InvalidRequest as e:
    if "path is already in use" in str(e):
        print("Le moteur est déjà activé. Poursuite de la configuration...")
    else:
        raise

# 2. Configuration stricte de l'historique et du Check-And-Set (CAS)
client.secrets.kv.v2.configure(max_versions=20, cas_required=True, mount_point="shared")
print("Configuration de l'historique et du CAS (Check-And-Set) appliquée avec succès.")

Moteur KV-v2 activé sur le chemin 'shared/'.
Configuration de l'historique et du CAS (Check-And-Set) appliquée avec succès.


Le paramètre `cas_required=True` est crucial pour éviter la perte de données concurrentes. Lorsqu'il est actif, le client doit prouver qu'il connaît la version actuelle du secret avant de la modifier. Si un développeur tente de mettre à jour un secret alors que la valeur du paramètre cas fournie ne correspond pas à la version courante sur le serveur, l'API lèvera une exception `hvac.exceptions.InvalidRequest`. Si un secret est créé pour la première fois, le paramètre cas doit être fixé à 0.

### 2. Écriture, Extraction et Métadonnées

L'architecture sépare conceptuellement les données du secret de ses métadonnées et de sa structure. Les développeurs peuvent interroger l'arborescence sans exposer les données via l'endpoint `/subkeys/`. Lors de l'appel à cet endpoint, Vault récupère les secrets, mais remplace la valeur sous-jacente des clés feuille (non-map) par une valeur `null` (ou `nil` selon le client).


In [ ]:
# Création initiale sécurisée d'un secret (cas=0 car le secret n'existe pas encore)
client.secrets.kv.v2.create_or_update_secret(
    mount_point="shared",
    path="dev/square-api",
    secret={"prod": "5678", "sandbox": "1234"},
    cas=0,
)
print("Secret 'dev/square-api' créé avec succès.")

# Lecture complète de la version la plus récente
secret_resp = client.secrets.kv.v2.read_secret_version(
    mount_point="shared", path="dev/square-api"
)

# Extraction des informations pertinentes
print(f"Clés disponibles dans le secret : {secret_resp['data']['data'].keys()}")
print(f"Date de création : {secret_resp['data']['metadata']['created_time']}")
print(f"Version actuelle du secret : {secret_resp['data']['metadata']['version']}")

Secret 'dev/square-api' créé avec succès.
Clés disponibles dans le secret : dict_keys(['prod', 'sandbox'])
Date de création : 2026-08-13T07:58:11.549180403Z
Version actuelle du secret : 1


/tmp/ipykernel_90974/2981975496.py:11: DeprecationWarning: The raise_on_deleted_version parameter will change its default value to False in hvac v3.0.0. The current default of True will preserve previous behavior. To use the old behavior with no warning, explicitly set this value to True. See https://github.com/hvac/hvac/pull/907
  secret_resp = client.secrets.kv.v2.read_secret_version(


Une fois que tu as vérifié que la version actuelle retournée est bien la version 1.

Si un second développeur de ton équipe tente de modifier le mot de passe prod de ce même secret en exécutant exactement le même script (avec cas=0)
cela causera un echec.

**Vault est l'unique source de vérité et centralise tout l'état de l'infrastructure**. Si la requête échoue, c'est grâce au mécanisme **Check-And-Set (CAS)** que nous avons imposé côté serveur via le paramètre `cas_required=True`.

Voici la mécanique exacte :

- En envoyant cas=0, le script dit formellement à Vault : "Crée ce secret uniquement s'il n'a jamais existé".
- Puisque ton secret existe maintenant et se trouve à la version 1, Vault va bloquer la transaction et lever une exception `hvac.exceptions.InvalidRequest`.
- Pour que le second développeur puisse modifier le mot de passe prod, il devra impérativement lire le secret d'abord, constater qu'il est en version 1, puis envoyer sa mise à jour avec cas=1.C'est ce qui empêche deux systèmes ou développeurs d'écraser leurs modifications respectives de manière concurrente.

I​l​ ​est​ ​également​ ​possible​ ​d'implémenter​ ​une​ ​politique​ ​centralisée​ ​de​ ​génération​ ​de​ ​mots​ ​de​ ​passe​ ​pour​ ​que​ ​les​ développeurs​ ​obtiennent​ ​des​ ​chaînes​ ​aléatoires​ ​conformes.​ ​La​ ​politique​ ​HCL​ ​est​ ​envoyée​ ​via​ ​une​ ​requête​ ​POST​ ​au​ ​chemin​ `​sys/policies/password/:nom​​`.​ ​Les​ ​règles​ ​peuvent​ ​imposer​​ la ​longueur,​​et ​​un ​​minimum​ ​de ​​caractères ​​minuscules, ​​majuscules ​de ​​chiffres​ ​et ​​de ​​symboles ​​spéciaux​ ​(via​ ​la déclaration​​ `rule "charset" { min-chars = X }`​​). L'endpoint ne retourne aucune donnée en cas de succès.​


### 3. Exploration et Sous-clés (Subkeys)

L'architecture de Vault sépare conceptuellement les données du secret de ses métadonnées.  
Voici comment interroger l'arborescence des secrets sans jamais exposer les mots de passe en clair sur le réseau en utilisant l'endpoint `/subkeys/`. Cette fois-ci, nous n'utilisons pas `hvac`, mais une requête HTTP directe avec la bibliothèque.


In [ ]:
import os
import requests

# L'API remplace la valeur sous-jacente des clés feuille par 'null' pour les masquer
subkeys_endpoint = f"{os.getenv('VAULT_ADDR')}/v1/shared/subkeys/dev/square-api"

subkeys_resp = requests.get(
    subkeys_endpoint, headers={"X-Vault-Token": os.getenv("VAULT_TOKEN")}
)

print("Exploration de la structure (sans les valeurs) :")
print(subkeys_resp.json()["data"]["subkeys"])

Exploration de la structure (sans les valeurs) :
{'prod': None, 'sandbox': None}


l'objectif est de connaître la structure. Mais poussons l'analyse de cette mécanique un peu plus loin.

Pourquoi concevoir un endpoint spécifique au lieu de simplement demander aux développeurs de lire le secret et d'ignorer les valeurs ? La réponse se trouve dans les journaux d'audit (audit logs).

Si un script de CI/CD ou une interface graphique lit le secret complet juste pour vérifier que la clé `prod` existe, Vault enregistre un accès aux données sensibles. Si ce script tourne toutes les 5 minutes, vos logs de sécurité seront saturés de faux positifs d'accès. En utilisant l'endpoint `/subkeys/`, le système prouve que la clé existe tout en remplaçant la valeur par null, ce qui permet de valider la conformité d'une structure de données sans déclencher d'alerte de sécurité pour lecture de mot de passe.

### 4. Politique Centralisée de Mots de Passe

Pour clôturer le dernier mécanisme de contrôle : la génération de mots de passe dictée par le serveur.  
Plutôt que de laisser chaque microservice générer ses propres chaînes aléatoires (avec le risque d'utiliser de mauvais algorithmes), on peut configurer une politique HCL stricte via l'API `sys/policies/password/:nom`.


In [ ]:
import os
import requests

# Définition de la politique HCL (HashiCorp Configuration Language)
password_policy = """
length = 20
rule "charset" {
  charset = "abcdefghijklmnopqrstuvwxyz"
  min-chars = 2
}
rule "charset" {
  charset = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
  min-chars = 2
}
rule "charset" {
  charset = "0123456789"
  min-chars = 2
}
rule "charset" {
  charset = "!@#$%^&*"
  min-chars = 2
}
"""

policy_endpoint = f"{os.getenv('VAULT_ADDR')}/v1/sys/policies/password/norme-entreprise"

# L'endpoint ne retourne aucune donnée (204 No Content) en cas de succès
response = requests.post(
    policy_endpoint,
    headers={"X-Vault-Token": os.getenv("VAULT_TOKEN")},
    json={"policy": password_policy},
)

if response.status_code == 204:
    print("Politique de mot de passe 'norme-entreprise' configurée avec succès.")
else:
    print(f"Échec de la configuration : {response.status_code} - {response.text}")

Politique de mot de passe 'norme-entreprise' configurée avec succès.


## ​Niveau 2 : Intermédiaire - Automatisation M2M avec AppRole​

​Déployer​ ​des​ ​jetons​ ​Vault​ ​en​ ​dur​ ​dans​ ​le​ ​code​ ​d'une​ ​application​ ​est​ ​une​ ​faille​ ​de​ ​sécurité​ majeure.​ ​AppRole​ ​est​ ​la​ ​méthode​ ​d'authentification​ ​privilégiée​ ​pour​ ​les​ ​architectures​ ​"Machine-to-Machine".​​ Elle​ ​repose ​​sur​ ​deux​ ​composantes​​: ​​le​ `​RoleID`​ ​(comparable​ ​à​ ​un​​ login​ public) ​​et​​le​ `​SecretID` (mot de passe éphémère à usage unique).​

​Le​ ​flux​ ​sécurisé​ ​exige​ ​l'utilisation​ ​d'un​ ​mécanisme​ ​de​ ​"Pull",​ ​où​ ​le​ ​système​ ​de​ ​déploiement​ ​pousse​ ​uniquement​ ​le​ ​RoleID​ ​dans​ ​la​ ​machine​ ​cible,​ ​tandis​ ​que​ ​l'application​ ​"tire"​ ​le​ ​SecretID​ ​d'une​ ​source​ ​sécurisée​ ​au​ ​démarrage.​ ​Le​ ​mode​ ​"Push"​ ​(où​ ​un​ ​tiers​ ​génère​ ​le​ ​jeton​ ​final​ ​et​ ​le​ ​pousse​ ​à​ ​l'application)​ ​est​ ​déconseillé​ ​car​ ​il​ ​oblige​ ​ce​ ​système​ ​tiers​ ​à​ ​manipuler​ ​directement​ ​les​ ​informations​ ​d'identification finales.​

### 1. Configuration du Rôle Applicatif


In [8]:
import hvac

# 1. Activation de la méthode d'authentification AppRole (avec idempotence)
try:
    client.sys.enable_auth_method(method_type="approle", path="approle")
    print("Méthode AppRole activée.")
except hvac.exceptions.InvalidRequest as e:
    if "path is already in use" in str(e):
        print("AppRole est déjà activé.")
    else:
        raise

# 2. Création du rôle applicatif avec les contraintes du document
client.auth.approle.create_or_update_approle(
    role_name="backend-microservice",
    bind_secret_id=True,  # Force la présentation du SecretID lors du login
    secret_id_num_uses=1,  # Le SecretID ne peut être utilisé qu'une seule fois
    secret_id_ttl="10m",  # Le SecretID périme 10 minutes après sa création
    secret_id_bound_cidrs=[
        "127.0.0.1/32"
    ],  # Restreint la demande de jeton au réseau interne
    token_policies=["default"],
)
print("Rôle 'backend-microservice' configuré avec ses contraintes de sécurité.")

AppRole est déjà activé.
Rôle 'backend-microservice' configuré avec ses contraintes de sécurité.


En ingénierie de la sécurité, cela répond au **Principe de Moindre Privilège**. Si ton application est compromise (par exemple, via une faille d'injection qui permet à un attaquant d'exécuter du code), l'attaquant hérite des droits de l'application.

Si ton application possédait la permission de lire son propre RoleID ou de générer de nouveaux SecretID, l'attaquant pourrait s'en servir pour forger ses propres accès persistants, totalement indépendants du cycle de vie de ton application. En forçant l'application à ne connaître que l'endpoint de connexion (`/auth/approle/login`), tu garantis qu'une compromission applicative ne se transforme pas en compromission de l'infrastructure d'authentification.


### 2. Authentification Autonome (AppRole)

Maintenant que le rôle est correctement configuré pour accepter les requêtes de ta machine, nous allons simuler le flux d'authentification complet.

Cette étape se divise en deux phases distinctes telles que décrites dans le document :

- Le rôle de l'Administrateur (ou CI/CD) : Il extrait le RoleID (qui ne change pas) et génère un SecretID éphémère (qui périme après 10 minutes et 1 utilisation). Il peut d'ailleurs y attacher des métadonnées pour faciliter l'audit opérationnel.
- Le rôle de l'Application : Elle utilise un nouveau client hvac totalement vierge (sans le jeton d'administration) et s'authentifie de manière autonome en présentant ce couple d'identifiants.


In [ ]:
import os
import hvac

# --- PARTIE 1 : SIMULATION DE L'ADMINISTRATEUR ---

# Extraction du RoleID (identifiant public)
role_id_resp = client.auth.approle.read_role_id(role_name="backend-microservice")
role_id = role_id_resp["data"]["role_id"]

# Génération dynamique du SecretID (Correction : utilisation d'un dictionnaire Python)
secret_id_resp = client.auth.approle.generate_secret_id(
    role_name="backend-microservice",
    metadata={"env": "production", "region": "eu-west"},
)
secret_id = secret_id_resp["data"]["secret_id"]

print(f"RoleID extrait : {role_id[:8]}...")
print(f"SecretID généré (usage unique) : {secret_id[:8]}...")


# --- PARTIE 2 : SIMULATION DE L'APPLICATION ---

# L'application initialise un nouveau client sans token initial
app_client = hvac.Client(url=os.getenv("VAULT_ADDR"))

# Authentification autonome via AppRole
app_client.auth.approle.login(role_id=role_id, secret_id=secret_id)

print(f"Application authentifiée de manière autonome : {app_client.is_authenticated()}")

RoleID extrait : d9ee4341...
SecretID généré (usage unique) : c507dc37...
Application authentifiée de manière autonome : True
